# 面试题：WordPiece 如何训练与编码，为什么不能只做最长字符串匹配？

## 面试回答主线

WordPiece 从字符级符号开始，反复把“共同出现得异常紧密”的相邻符号合成新词片。与只看 pair 次数的 BPE 不同，经典教学近似会用 `pair_freq / (left_freq × right_freq)` 衡量关联强度，避免两个本身极常见的符号仅凭边际频率霸榜。训练产物不仅是词表，还包括首词片与续接词片的约定，例如 `play + ##ing`。推理阶段对每个预切分 word 执行 longest-match-first；若任一位置无法覆盖，通常整个 word 退化为 `[UNK]`。因此规范化、预切分、基础字符覆盖和最大单词长度都是 tokenizer 合同的一部分，而不只是词表细节。

## 真实案例：英文电子配件站内搜索

数据是带查询次数的脱敏教学日志，覆盖 `wire/charge/play` 等可复用词干及其后缀。真实 WordPiece 会在数百万条语料上训练更大词表；这里的小样本只用于把 score、merge 和推理路径完整打印出来。

In [1]:
from collections import Counter  # 导入计数器以统计 token 和相邻 pair 的加权频次。
import re  # 导入正则表达式以实现修复后的预切分器。
training_queries = [  # 构造带业务频次的电子配件搜索日志。
    ("wireless", 160),  # 高频无线属性词提供可复用的字母片段。
    ("wired", 100),  # 同词干的不同时态帮助学习 wire 子词。
    ("wiring", 70),  # 共享 wir 前缀并带 ing 后缀。
    ("charger", 150),  # 高频充电器品类提供 charge 词干。
    ("charging", 130),  # 共享 charg 并带 ing 后缀。
    ("rechargeable", 80),  # 前缀与后缀组合检验子词复用能力。
    ("player", 60),  # player 与 playing 共享 play 词干。
    ("playing", 55),  # ing 后缀可跨多个词干复用。
    ("headphones", 90),  # 较长品类词检验压缩效果。
]  # 结束训练查询列表。
print("训练查询           频次  字符数")  # 输出真实案例输入表标题。
for query, frequency in training_queries:  # 逐条展示训练词及业务权重。
    print(f"{query:<18} {frequency:>4} {len(query):>7}")  # 输出查询文本、频次和字符长度。
print("总查询次数：", sum(frequency for _, frequency in training_queries))  # 输出加权语料规模以解释统计口径。

训练查询           频次  字符数
wireless            160       8
wired               100       5
wiring               70       6
charger             150       7
charging            130       8
rechargeable         80      12
player               60       6
playing              55       7
headphones           90      10
总查询次数： 895


## Baseline：只保留最高频整词

固定容量下，整词词表对训练内热门查询很省 token，但任何新词形都会整体变成 `[UNK]`。下面只保留五个最高频整词，并用包含新组合的查询集观察覆盖率。

In [2]:
whole_word_vocabulary = {query for query, _ in sorted(training_queries, key=lambda item: (-item[1], item[0]))[:5]}  # 只保留五个最高频整词模拟固定容量词表。
evaluation_words = ["wireless", "rewiring", "chargers", "wirelesscharger", "replaying", "headphones"]  # 构造训练内、新词形与复合词混合的评测集。
def whole_word_encode(word):  # 定义整词查表基线编码器。
    return [word] if word in whole_word_vocabulary else ["[UNK]"]  # 词表外查询整体退化为未知词。
baseline_unknown = 0  # 统计整词基线产生未知词的样本数。
print("查询                 整词基线输出")  # 输出基线逐样本结果表标题。
for word in evaluation_words:  # 遍历固定评测集以保持后续比较公平。
    tokens = whole_word_encode(word)  # 对当前查询执行整词查表。
    baseline_unknown += int(tokens == ["[UNK]"])  # 累加未知词样本数量。
    print(f"{word:<20} {tokens}")  # 输出当前查询的真实基线结果。
print(f"整词基线 OOV：{baseline_unknown}/{len(evaluation_words)} = {baseline_unknown / len(evaluation_words):.1%}")  # 输出整词方案的样本级未知率。

查询                 整词基线输出
wireless             ['wireless']
rewiring             ['[UNK]']
chargers             ['[UNK]']
wirelesscharger      ['[UNK]']
replaying            ['[UNK]']
headphones           ['headphones']
整词基线 OOV：4/6 = 66.7%


## 核心实现：关联分数驱动的 WordPiece merge

首字符保持原样，后续字符加 `##`，这样词表能区分 `ing` 作为词首还是续接片段。每轮重新计算加权 token 频次和 pair 频次，再用关联分数选出唯一 merge；同分时依次比较 pair 次数和字典序，保证训练可复现。

In [3]:
def initial_symbols(word):  # 把一个单词转换为 WordPiece 的初始字符符号。
    return tuple(character if index == 0 else "##" + character for index, character in enumerate(word.lower()))  # 为非首字符添加续接标记。
def wordpiece_statistics(sequences):  # 统计当前切分下 token 与相邻 pair 的加权频次。
    token_counts = Counter()  # 创建 token 边际频次计数器。
    pair_counts = Counter()  # 创建相邻 pair 联合频次计数器。
    for sequence, frequency in sequences.items():  # 遍历每种切分序列及其业务权重。
        for token in sequence:  # 遍历当前序列中的每个 token。
            token_counts[token] += frequency  # 按查询次数累计 token 边际频次。
        for index in range(len(sequence) - 1):  # 枚举当前序列的全部相邻位置。
            pair_counts[(sequence[index], sequence[index + 1])] += frequency  # 按查询次数累计 pair 联合频次。
    return token_counts, pair_counts  # 返回计算关联分数所需的两组统计量。
def join_wordpieces(left, right):  # 按 WordPiece 的首词片与续接约定合并一对 token。
    right_surface = right[2:] if right.startswith("##") else right  # 去掉右 token 的续接前缀后拼接表面文本。
    return left + right_surface  # 保留左 token 是否以 ## 开头的边界属性。
def merge_wordpiece_sequence(sequence, target_pair):  # 在一条序列中非重叠地应用一个 WordPiece merge。
    merged = []  # 创建列表保存合并后的 token 序列。
    index = 0  # 从当前序列的首 token 开始扫描。
    while index < len(sequence):  # 持续扫描直到消费全部 token。
        can_merge = index + 1 < len(sequence) and (sequence[index], sequence[index + 1]) == target_pair  # 判断当前位置是否命中目标 pair。
        if can_merge:  # 命中目标 pair 时生成一个新词片。
            merged.append(join_wordpieces(sequence[index], sequence[index + 1]))  # 拼接两个 token 并保留续接语义。
            index += 2  # 一次跨过已经合并的两个 token。
        else:  # 未命中时原样保留当前 token。
            merged.append(sequence[index])  # 把当前 token 追加到新序列。
            index += 1  # 向后移动一个 token 继续扫描。
    return tuple(merged)  # 返回不可变序列以便作为字典键。
def train_wordpiece(rows, merge_steps):  # 从字符级符号开始训练指定轮数的 WordPiece。
    sequences = {initial_symbols(word): frequency for word, frequency in rows}  # 创建训练词到加权切分状态的映射。
    vocabulary = {token for sequence in sequences for token in sequence}  # 把全部基础字符符号加入固定词表。
    history = []  # 保存每轮 score、频次和 merge 结果供教学观察。
    for step in range(merge_steps):  # 每轮只新增一个关联最强的词片。
        token_counts, pair_counts = wordpiece_statistics(sequences)  # 重新统计当前边界下的边际与联合频次。
        if not pair_counts:  # 所有训练词都已经合为单 token 时停止。
            break  # 跳出训练循环并返回已有词表。
        scored_pairs = []  # 创建候选列表保存 pair 的关联分数。
        for pair, pair_count in pair_counts.items():  # 遍历本轮所有相邻 pair。
            denominator = token_counts[pair[0]] * token_counts[pair[1]]  # 计算左右 token 边际频次乘积。
            score = pair_count / denominator  # 使用经典教学近似衡量超出独立假设的关联强度。
            scored_pairs.append((score, pair_count, pair))  # 保存排序所需的完整候选信息。
        best_score, best_count, best_pair = sorted(scored_pairs, key=lambda item: (-item[0], -item[1], repr(item[2])))[0]  # 用稳定规则选择本轮唯一赢家。
        new_piece = join_wordpieces(best_pair[0], best_pair[1])  # 构造将被加入词表的新词片。
        updated = {}  # 创建映射保存应用本轮 merge 后的全部训练序列。
        for sequence, frequency in sequences.items():  # 对每个训练词应用同一个全局 merge。
            new_sequence = merge_wordpiece_sequence(sequence, best_pair)  # 更新当前词的 token 边界。
            updated[new_sequence] = updated.get(new_sequence, 0) + frequency  # 聚合可能相同的新序列及权重。
        vocabulary.add(new_piece)  # 把新词片加入推理阶段可用的词表。
        history.append((step + 1, best_pair, best_count, best_score, new_piece))  # 记录完整训练决策供解释。
        sequences = updated  # 让下一轮基于新边界重新统计。
    vocabulary.update({"[UNK]", "[PAD]"})  # 加入推理和批处理需要的特殊 token。
    return vocabulary, history, sequences  # 返回词表、逐轮轨迹和最终训练切分。
wordpiece_vocabulary, merge_history, final_sequences = train_wordpiece(training_queries, 45)  # 训练足够多轮以形成可复用词干和后缀。
print("轮次 | pair                 | pair频次 | 关联分数   | 新词片")  # 输出训练中间量表标题。
for step, pair, count, score, new_piece in merge_history[:15]:  # 展示前十五轮以观察关联分数如何驱动合并。
    print(f"{step:>4} | {pair[0] + ' + ' + pair[1]:<20} | {count:>8} | {score:>10.6f} | {new_piece}")  # 输出本轮 pair、频次、score 与新词片。
print("最终词表大小：", len(wordpiece_vocabulary), "，已学习 merge 数：", len(merge_history))  # 输出容量和训练进度摘要。

轮次 | pair                 | pair频次 | 关联分数   | 新词片
   1 | ##d + ##p            |       90 |   0.005263 | ##dp
   2 | ##o + ##n            |       90 |   0.002899 | ##on
   3 | p + ##l              |      115 |   0.002817 | pl
   4 | ##b + ##l            |       80 |   0.004167 | ##bl
   5 | c + ##h              |      280 |   0.002222 | ch
   6 | ##dp + ##h           |       90 |   0.005882 | ##dph
   7 | ##c + ##h            |       80 |   0.012500 | ##ch
   8 | ##dph + ##on         |       90 |   0.011111 | ##dphon
   9 | w + ##i              |      330 |   0.001709 | wi
  10 | ##i + ##n            |      255 |   0.003922 | ##in
  11 | ##y + ##in           |       55 |   0.001876 | ##yin
  12 | ##in + ##g           |      200 |   0.001626 | ##ing
  13 | ##yin + ##g          |       55 |   0.002410 | ##ying
  14 | ##g + ##ing          |      130 |   0.001806 | ##ging
  15 | ch + ##a             |      280 |   0.001550 | cha
最终词表大小： 67 ，已学习 merge 数： 45


## 推理实现与结果表：longest-match-first 只在一个预切分 word 内运行

编码器从当前位置尝试最长候选，非首位置自动加 `##`。只要某个位置没有任何词片覆盖，整个 word 返回 `[UNK]`，这是 WordPiece 与任意字符回退策略的重要区别。

In [4]:
def encode_wordpiece(word, vocabulary, max_chars=100):  # 对单个预切分 word 执行 longest-match-first 编码。
    normalized = word.lower()  # 使用与训练一致的小写规范化视图。
    if len(normalized) > max_chars:  # 限制极长 word 以避免近似二次复杂度扫描。
        return ["[UNK]"]  # 超长输入直接走显式失败路径。
    tokens = []  # 创建列表保存最终词片。
    start = 0  # 从单词首字符开始寻找最长覆盖。
    while start < len(normalized):  # 持续编码直到覆盖完整单词。
        end = len(normalized)  # 每个位置先尝试延伸到词尾的最长候选。
        matched = None  # 初始化本位置尚未找到合法词片。
        while start < end:  # 逐步缩短候选直到命中词表。
            surface = normalized[start:end]  # 取出当前字符区间的表面字符串。
            candidate = surface if start == 0 else "##" + surface  # 非首位置使用续接词片命名约定。
            if candidate in vocabulary:  # 找到当前起点的最长合法词片时停止缩短。
                matched = candidate  # 保存命中的最长词片。
                break  # 跳出内层搜索并提交本词片。
            end -= 1  # 缩短右边界继续尝试更短候选。
        if matched is None:  # 任一位置无法覆盖就不能保持整个 word 的可逆切分。
            return ["[UNK]"]  # 按标准 WordPiece 行为把整词标为未知。
        tokens.append(matched)  # 把命中的最长词片加入结果。
        start = end  # 从刚刚覆盖区间的末尾继续编码。
    return tokens  # 返回完整覆盖该 word 的词片序列。
print("查询                 整词基线  WordPiece数  WordPiece切分")  # 输出相同评测集上的方案对比标题。
wordpiece_results = {}  # 保存逐词编码结果供失败分析和测试使用。
for word in evaluation_words:  # 对每个训练内或组合查询运行同一词表。
    tokens = encode_wordpiece(word, wordpiece_vocabulary)  # 执行真正的 longest-match-first 编码。
    wordpiece_results[word] = tokens  # 按原查询保存可读词片。
    print(f"{word:<20} {str(whole_word_encode(word)):<12} {len(tokens):>11}  {tokens}")  # 输出整词与 WordPiece 的逐样本对照。
wordpiece_unknown = sum(tokens == ["[UNK]"] for tokens in wordpiece_results.values())  # 统计 WordPiece 的样本级未知数量。
print(f"OOV 对照：整词={baseline_unknown}/{len(evaluation_words)}，WordPiece={wordpiece_unknown}/{len(evaluation_words)}")  # 输出同一评测集下的覆盖率改进。

查询                 整词基线  WordPiece数  WordPiece切分
wireless             ['wireless']           1  ['wireless']
rewiring             ['[UNK]']              1  ['[UNK]']
chargers             ['[UNK]']              2  ['charger', '##s']
wirelesscharger      ['[UNK]']              3  ['wireless', '##charge', '##r']
replaying            ['[UNK]']              6  ['r', '##e', '##p', '##l', '##a', '##ying']
headphones           ['headphones']           1  ['headphones']
OOV 对照：整词=4/6，WordPiece=1/6


## 结果解读

整词基线无法处理 `rewiring`、`chargers` 和 `replaying`，WordPiece 则能复用首词片、`##ing`、`##er` 等续接片段。关联分数与绝对 pair 频次回答不同问题：前者寻找“相对独立假设更紧密”的组合，后者偏向全局热门字符。实际实现会采用更高效的数据结构和更严谨的似然增益，但边际频次、pair 频次与稳定 tie-break 仍是面试时必须说清的中间量。

## 失败案例：把整条查询直接交给 WordPiece

WordPiece 假设输入已经过 pre-tokenization。训练语料没有连字符，把 `wireless-charger` 当成一个 word 会导致整个片段 `[UNK]`；修复不是随意把连字符塞进词表，而是先按字母数字与标点切分，再对普通 word 应用 WordPiece、对标点走显式 token。

In [5]:
raw_query = "wireless-charger"  # 构造线上常见的带连字符商品查询。
broken_encoding = encode_wordpiece(raw_query, wordpiece_vocabulary)  # 错误地把整条查询当成一个预切分 word。
def basic_pretokenize(text):  # 定义最小预切分器以分离单词、数字和标点。
    return re.findall(r"[A-Za-z0-9]+|[^\w\s]", text)  # 返回保持原顺序的词段与独立标点。
def encode_query(text, vocabulary):  # 定义包含预切分和 WordPiece 的完整查询编码管线。
    final_tokens = []  # 创建列表按原顺序汇总每个片段的 token。
    for segment in basic_pretokenize(text):  # 逐个处理预切分产生的词段或标点。
        if re.fullmatch(r"[A-Za-z0-9]+", segment):  # 普通字母数字段使用 WordPiece 编码。
            final_tokens.extend(encode_wordpiece(segment, vocabulary))  # 追加当前 word 的 WordPiece 结果。
        else:  # 标点不应污染 word 内部的最长匹配过程。
            final_tokens.append(f"[PUNCT:{segment}]")  # 使用显式可审计的标点 token 保留信息。
    return final_tokens  # 返回覆盖完整原查询的 token 序列。
fixed_encoding = encode_query(raw_query, wordpiece_vocabulary)  # 对同一查询应用预切分修复。
print("原始查询：", raw_query)  # 输出触发问题的业务输入。
print("未预切分：", broken_encoding, "，是否整词丢失：", broken_encoding == ["[UNK]"])  # 展示错误管线的未知词行为。
print("预切分结果：", basic_pretokenize(raw_query))  # 展示 WordPiece 之前真正需要的边界。
print("修复后编码：", fixed_encoding, "，未知词数量：", fixed_encoding.count("[UNK]"))  # 展示修复后保留词义和标点的结果。

原始查询： wireless-charger
未预切分： ['[UNK]'] ，是否整词丢失： True
预切分结果： ['wireless', '-', 'charger']
修复后编码： ['wireless', '[PUNCT:-]', 'charger'] ，未知词数量： 0


## 生产差距与落地清单

教学训练器只有九个聚合词，线上要在规范化后的大语料上做高效 pair 更新、词表预算控制和特殊 token 保护。必须共同版本化 normalizer、pre-tokenizer、`##` 约定、词表与最大单词长度；发布前按语言、渠道和产品类别比较 token/character 比、整词 `[UNK]` 率、P99 编码时延与下游指标。若业务必须无未知输入，可在 WordPiece 外加 byte fallback，但这会改变 token id 合同，不能静默上线。

## 最小回归测试

断言只固定核心行为，训练轨迹、逐查询对照和连字符失败修复才是主要学习内容。

In [6]:
assert merge_history == train_wordpiece(training_queries, 45)[1]  # 验证稳定 tie-break 使重复训练得到相同 merge 顺序。
assert wordpiece_results["wireless"] != ["[UNK]"]  # 验证训练内高频查询能够被完整覆盖。
assert wordpiece_unknown < baseline_unknown  # 验证子词复用相对固定整词词表降低样本级 OOV。
assert broken_encoding == ["[UNK]"]  # 固化缺少预切分时连字符导致整词失败的反例。
assert fixed_encoding.count("[UNK]") == 0  # 验证预切分修复后单词与标点都得到显式表示。
assert encode_wordpiece("a" * 101, wordpiece_vocabulary) == ["[UNK]"]  # 验证超长 word 门禁避免无界最长匹配扫描。
print("最小回归测试通过：关联训练、子词覆盖、预切分修复和长度门禁均符合预期。")  # 输出顺序执行完成的明确结论。

最小回归测试通过：关联训练、子词覆盖、预切分修复和长度门禁均符合预期。
